# 04 — Generador de oportunidades

**Checkpoint: Día 2 — revisión de consolidación.** El motor de reglas (`saberlink/opportunities.py`) nunca usa IA generativa — son condiciones if/else sobre las señales ya calculadas por `scoring.py`. Cada oportunidad carga la evidencia (IDs + score) que la sustenta.

In [1]:
import sys
sys.path.insert(0, '..')

from saberlink import pipeline

## Las 7 reglas implementadas

1. `RESEARCH_CONTINUITY` — antecedente PRJ/THS con estado válido (`ACTIVE`/`COMPLETED` para proyectos, `APPROVED` para tesis) + dominio y método afines.
2. `COLLABORATION` — 2+ grupos con cobertura complementaria (distinta facultad o dominios no solapados).
3. `CURRICULAR_INTEGRATION` — asignatura/competencia/resultado de aprendizaje afín en un programa activo.
4. `CAPABILITY_ACTIVATION` — capacidad institucional activa y madura, sin antecedente de proyecto/tesis fuerte.
5. `NEW_RESEARCH` — sin antecedente, pero investigador/grupo con dominio afín.
6. `KNOWLEDGE_TRANSFER` — antecedente relevante originado en otra facultad.
7. `THESIS_OPPORTUNITY` — tesis afín sin antecedente de proyecto fuerte, en un programa activo.

## Recorrido por varios casos, distintos dominios

In [2]:
for need_id in ['NEED-001', 'NEED-005', 'NEED-013', 'NEED-007']:
    out = pipeline.run_query(entity_id=need_id, top_k=5)
    print(f'--- {need_id}: {out["source"]["id"]} ---')
    if not out['opportunities']:
        print('  (ninguna regla disparó)')
    for o in out['opportunities']:
        print(f"  [{o['type']}] {o['opportunity']}")
        print(f"    razón: {o['reason']} | prioridad: {o['priority']}")
        print(f"    entidades relacionadas: {o['related_entities']}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- NEED-001: NEED-001 ---
  [RESEARCH_CONTINUITY] Continuar la línea de trabajo de PRJ-001 para atender NEED-001.
    razón: Antecedente con estado ACTIVE y dominio afín (y método afín, cuando aplica). | prioridad: media
    entidades relacionadas: ['NEED-001', 'PRJ-001', 'GRP-014']
  [CURRICULAR_INTEGRATION] Integrar COM-0052 al currículo para articular con NEED-001.
    razón: Componente curricular con dominio afín en un programa activo. | prioridad: media
    entidades relacionadas: ['NEED-001', 'COM-0052', 'PRG-004']



--- NEED-005: NEED-005 ---
  [RESEARCH_CONTINUITY] Continuar la línea de trabajo de PRJ-033 para atender NEED-005.
    razón: Antecedente con estado COMPLETED y dominio afín (y método afín, cuando aplica). | prioridad: media
    entidades relacionadas: ['NEED-005', 'PRJ-033', 'GRP-007']



--- NEED-013: NEED-013 ---
  [RESEARCH_CONTINUITY] Continuar la línea de trabajo de PRJ-097 para atender NEED-013.
    razón: Antecedente con estado ACTIVE y dominio afín (y método afín, cuando aplica). | prioridad: media
    entidades relacionadas: ['NEED-013', 'PRJ-097', 'GRP-011']



--- NEED-007: NEED-007 ---
  [RESEARCH_CONTINUITY] Continuar la línea de trabajo de THS-121 para atender NEED-007.
    razón: Antecedente con estado APPROVED y dominio afín (y método afín, cuando aplica). | prioridad: media
    entidades relacionadas: ['NEED-007', 'THS-121']



## Nota sobre el bug real que corrigió esto durante el build

Al probar `NEED-007` no disparaba ninguna oportunidad. La causa: las 650 tesis usan únicamente el estado `APPROVED` (nunca `ACTIVE`/`COMPLETED`), así que el filtro de estado excluía el 100% de las tesis como antecedente válido, sin importar la consulta. Corregido con `VALID_ANTECEDENT_STATUS` diferenciado por tipo de entidad.